In [1]:
def calcular_promedio(datos):
    if len(datos) == 0:
        return 0
    return sum(datos) / len(datos)


def calcular_humedad(datos):
    if len(datos) == 0:
        return 0
    return sum(datos) / len(datos)


def calcular_variacion(datos):
    if len(datos) < 2:
        return 0
    return datos[-1] - datos[0]


def resumen_compostaje(temperaturas, humedades):
    promedio_temp = calcular_promedio(temperaturas)
    promedio_humedad = calcular_humedad(humedades)
    variacion_temp = calcular_variacion(temperaturas)

    print("===== RESUMEN DEL COMPOSTAJE =====")
    print("Temperatura promedio:", promedio_temp, "°C")
    print("Humedad promedio:", promedio_humedad, "%")
    print("Variación de temperatura:", variacion_temp, "°C")

In [2]:
DIAS = 30            # duración de la simulación
T_AMBIENTE = 18      # temperatura ambiente (°C)

# Cada lote: masa seca, carbono, nitrógeno y agua (kg) y temperatura (°C)
lotes = {
    "Lote A": {"seca": 30.0, "carbono": 13.8, "nitrogeno": 0.58, "agua": 45.0, "temp": T_AMBIENTE},
    "Lote B": {"seca": 30.0, "carbono": 12.9, "nitrogeno": 0.45, "agua": 45.0, "temp": T_AMBIENTE},
    "Lote C": {"seca": 30.0, "carbono": 13.5, "nitrogeno": 0.84, "agua": 45.0, "temp": T_AMBIENTE},
}

# Aquí se guarda la evolución de cada lote, día por día
historial = {nombre: [] for nombre in lotes}

In [3]:
for dia in range(1, DIAS + 1):                # bucle 1: cada día
    for nombre, lote in lotes.items():        # bucle 2: cada lote en paralelo
        humedad = lote["agua"] / (lote["agua"] + lote["seca"])
        cn = lote["carbono"] / lote["nitrogeno"]

        # Factores entre 0 y 1: qué tan favorables son las condiciones
        f_temp = max(0, 1 - ((lote["temp"] - 55) / 45) ** 2)       # ideal cerca de 55 °C
        f_hum = max(0, 1 - ((humedad - 0.55) / 0.40) ** 2)         # ideal cerca de 55 %
        f_cn = min(1, 30 / cn)                                     # ideal C/N cerca de 30 o menos

        tasa = 0.03 * f_temp * f_hum * f_cn      # fracción de carbono que se degrada hoy

        c_perdido = lote["carbono"] * tasa
        lote["carbono"] -= c_perdido                                 # se pierde carbono (CO2)
        lote["seca"] -= c_perdido / 0.5                              # y con él, masa seca
        lote["agua"] -= lote["agua"] * 0.012 * max(0, lote["temp"] - 15) / 40   # evaporación
        lote["temp"] += 0.35 * ((T_AMBIENTE + 1500 * tasa) - lote["temp"])      # el lote se calienta

        historial[nombre].append({
            "dia": dia,
            "temp": round(lote["temp"], 1),
            "humedad": round(humedad * 100, 1),
            "cn": round(cn, 1),
            "masa": round(lote["seca"] + lote["agua"], 1),
        })

In [4]:
for nombre in historial:                                   # bucle 3: un resumen por lote
    temperaturas = [d["temp"] for d in historial[nombre]]
    humedades = [d["humedad"] for d in historial[nombre]]
    print("\n>>>", nombre)
    resumen_compostaje(temperaturas, humedades)


>>> Lote A
===== RESUMEN DEL COMPOSTAJE =====
Temperatura promedio: 54.93333333333333 °C
Humedad promedio: 64.30333333333334 %
Variación de temperatura: 35.3 °C

>>> Lote B
===== RESUMEN DEL COMPOSTAJE =====
Temperatura promedio: 55.190000000000005 °C
Humedad promedio: 63.70666666666667 %
Variación de temperatura: 36.0 °C

>>> Lote C
===== RESUMEN DEL COMPOSTAJE =====
Temperatura promedio: 55.02333333333333 °C
Humedad promedio: 64.09 %
Variación de temperatura: 35.5 °C
